# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


In [16]:
# Shared CVD-safe palette (Okabe–Ito based; no red/green pairs)
AFRICA = '#E69F00'   # accent: orange – the same continent keeps the same colour in every chart
GREY   = '#B8B8B8'   # context
CONT_COLORS = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
               'North America': '#56B4E9', 'Oceania': '#7F7F7F', 'South America': '#8C6D31'}

---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** **Line chart, indexed to 1970 = 100.**

- The question is about *change over time*, so a line chart is the natural choice: slope = growth speed, and the eye can follow each continent from start to finish. The x-axis is numeric, so the unequal gaps between the census years (10 → 5 → 2 years) are drawn honestly.
- Indexing every continent to 1970 = 100 makes continents of very different size comparable ("relative to its 1970 size"). Raw populations would squash Oceania and Europe flat against Asia.
- Colour encodes meaning: **Africa is the single orange accent** (the answer), every other continent is grey context. Lines are labelled directly at the right end instead of using a legend, and gridlines are removed.

In [5]:
pop = (df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
              .sort_values(['Continent', 'Year']))
base = pop[pop['Year'] == 1970].set_index('Continent')['Population']
pop['Index'] = pop['Population'] / pop['Continent'].map(base) * 100

end = pop[pop['Year'] == 2022].set_index('Continent')['Index'].sort_values(ascending=False)
fastest = end.index[0]

fig = px.line(pop, x='Year', y='Index', color='Continent', markers=True,
              color_discrete_map={c: (AFRICA if c == 'Africa' else GREY) for c in end.index},
              template='plotly_white')
fig.update_traces(line_width=2, marker_size=5)
fig.update_traces(line_width=4, marker_size=7, selector=dict(name='Africa'))

# direct labels at the end of each line (nudged apart so they do not overlap)
min_gap, last_y = 13, None
for c, v in end.items():
    y = v if last_y is None or last_y - v >= min_gap else last_y - min_gap
    last_y = y
    fig.add_annotation(x=2022, y=y, xshift=10, xanchor='left', showarrow=False,
                       text=f"<b>{c}</b> {v:.0f}" if c == fastest else f"{c} {v:.0f}",
                       font=dict(color='#B07800' if c == fastest else '#666', size=13 if c == fastest else 12))

fig.add_hline(y=100, line_dash='dot', line_color='#999', line_width=1)
fig.update_layout(
    title=f"{fastest}'s population has grown {end[fastest]/100:.1f}× since 1970 – "
          f"far faster than any other continent<br><sup>Population by continent, indexed to 1970 = 100</sup>",
    showlegend=False, margin=dict(r=140))
fig.update_xaxes(title=None, showgrid=False, tickvals=[1970, 1980, 1990, 2000, 2010, 2022], range=[1968, 2023])
fig.update_yaxes(title=None, showgrid=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** **Sorted horizontal bar chart (top 15).**

- The task is a *ranking of a quantity* (people added, 2022 − 1970) across discrete countries, so a bar chart sorted by value is the clearest choice: bar length is the most accurate visual encoding and the ranking can be read top to bottom.
- Horizontal bars give long country names room to be read without rotating text.
- One accent colour (orange = Africa) answers the second half of the question at a glance; the other continents are grey. Values are written at the bar ends, so the value axis and the legend are not needed and are removed.

In [7]:
gain = df.assign(Gain=df['2022 Population'] - df['1970 Population']).nlargest(15, 'Gain').sort_values('Gain')
gain['Group'] = np.where(gain['Continent'] == 'Africa', 'Africa', 'Other continents')
n_africa = int((gain['Continent'] == 'Africa').sum())
n_asia = int((gain['Continent'] == 'Asia').sum())

fig = px.bar(gain, x='Gain', y='Country', orientation='h', color='Group',
             color_discrete_map={'Africa': AFRICA, 'Other continents': GREY},
             text=gain['Gain'].map(lambda v: f'{v/1e6:,.0f}M'), template='plotly_white')
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_layout(
    title=f"{n_africa} of the 15 countries that added the most people since 1970 are in "
          f"<span style='color:#B07800'>Africa</span> – Asia has {n_asia}<br>"
          f"<sup>Population added between 1970 and 2022, top 15 countries (Africa highlighted)</sup>",
    showlegend=False, margin=dict(r=60))
fig.update_xaxes(visible=False)
fig.update_yaxes(title=None, showgrid=False)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** **Scatter plot (log–log), coloured by continent.**

- Two numeric variables and a question about their *relationship* → scatter plot (one dot per country).
- Both variables span many orders of magnitude (Vatican City to Russia / China), so both axes are logarithmic; on linear axes nearly all countries would collapse into the bottom-left corner. A dashed trend line shows the overall pattern, so deviations are easy to spot.
- Colour is used only for the third variable the question asks about – continent – with a colour-blind-safe palette. Only the countries furthest from the trend line (largest residuals) are labelled directly; everything else is available on hover.

In [8]:
d = df.copy()
lx, ly = np.log10(d['Area']), np.log10(d['2022 Population'])
slope, intercept = np.polyfit(lx, ly, 1)
d['resid'] = ly - (slope * lx + intercept)
r2 = np.corrcoef(lx, ly)[0, 1] ** 2

# label the biggest rule-breakers (most above / below the trend) + the two giants
label = set(d.nlargest(4, 'resid')['Country']) | set(d.nsmallest(4, 'resid')['Country']) | {'China', 'India'}
d['Label'] = np.where(d['Country'].isin(label), d['Country'], '')

fig = px.scatter(d, x='Area', y='2022 Population', color='Continent', text='Label',
                 hover_name='Country', log_x=True, log_y=True,
                 color_discrete_map=CONT_COLORS, template='plotly_white')
fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='white')),
                  textposition='top center', textfont_size=11)

xs = np.linspace(lx.min(), lx.max(), 50)
fig.add_scatter(x=10 ** xs, y=10 ** (slope * xs + intercept), mode='lines',
                line=dict(color='#444', dash='dash', width=1.5), hoverinfo='skip', showlegend=False)

top_pos = ' and '.join(d.nlargest(2, 'resid')['Country'])
top_neg = ' and '.join(d.nsmallest(2, 'resid')['Country'])
fig.update_layout(
    title=f"Bigger countries do tend to have more people (R² = {r2:.2f} on log scales), but "
          f"{top_pos} are far too crowded and {top_neg} far too empty<br>"
          f"<sup>Land area vs 2022 population, one dot per country, dashed line = overall trend</sup>",
    legend=dict(orientation='h', y=-0.15, title=None))
fig.update_xaxes(title='Land area (km², log scale)', showgrid=False)
fig.update_yaxes(title='Population 2022 (log scale)', showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** **Box plot with every country shown as a point, one box per continent (log y-axis).**

- The question asks for the *distribution* (spread, not the average) within and across groups → box plot (or violin). A box plot shows median, quartiles and outliers in a compact way that makes continents easy to compare side by side.
- Overlaying all countries as jittered points ("points='all'") shows the real data behind the boxes, which matters with only 14–57 countries per continent.
- Density ranges from <1 to >20,000 people/km², so a log axis is required; on a linear axis everything except a few city-states would be flat at zero.
- Continents are sorted by median density and use the same colours as the other charts. Median values and the most extreme countries are labelled; legend and vertical gridlines are removed because the x-axis already names the groups.

In [9]:
order = df.groupby('Continent')['Density'].median().sort_values().index.tolist()
med = df.groupby('Continent')['Density'].median()

fig = px.box(df, x='Continent', y='Density', color='Continent', points='all', log_y=True,
             hover_name='Country', category_orders={'Continent': order},
             color_discrete_map=CONT_COLORS, template='plotly_white')
fig.update_traces(jitter=0.5, pointpos=0, marker=dict(size=4, opacity=0.55), boxmean=False)

# median labels
for c in order:
    fig.add_annotation(x=c, y=np.log10(med[c]), text=f"median {med[c]:,.0f}", xshift=62,
                       showarrow=False, font=dict(size=11, color='#333'))
# most extreme countries
for _, r in df.nlargest(3, 'Density').iterrows():
    fig.add_annotation(x=r['Continent'], y=np.log10(r['Density']), text=r['Country'], xshift=-8,
                       xanchor='right', showarrow=False, font=dict(size=11, color='#333'))

spread = df.groupby('Continent')['Density'].agg(['min', 'max'])
widest = spread['max'].idxmax()
fig.update_layout(
    title=f"Density varies more <i>within</i> continents than between them – "
          f"{widest} runs from {spread.loc[widest, 'min']:,.0f} to {spread.loc[widest, 'max']:,.0f} people/km²<br>"
          f"<sup>Population density 2022 by continent, one dot per country (log scale)</sup>",
    showlegend=False)
fig.update_xaxes(title=None, showgrid=False)
fig.update_yaxes(title='People per km² (log scale)', showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** **Choropleth map of % change 2000 → 2022 with a diverging colour scale centred on 0.**

- The question has an explicit "where in the world" component, and the data is one value per country with an ISO-3 code → choropleth map.
- Growth and shrinkage are two sides of a meaningful midpoint (0 %), so a **diverging** scale is used with its neutral colour at exactly 0: **orange = shrank, blue = grew**. Orange/blue is colour-blind safe (no red/green).
- A few countries grew by more than 200 %, which would wash out the rest of the map, so colours are capped at ±80 %; the hover text still shows the true value. The map exposes the geographic pattern directly: shrinkage clusters in Eastern Europe and the Balkans/Baltics (plus some small island territories), growth is strongest across Africa and the Middle East.
- No legend clutter: only a slim colour bar, no frame or coastlines.

In [10]:
d = df.assign(Change=(df['2022 Population'] / df['2000 Population'] - 1) * 100)
d['Shown'] = d['Change'].clip(-80, 80)          # cap so extreme growers do not wash out the map
n_shrink = int((d['Change'] < 0).sum())
n_eu = int(((d['Change'] < 0) & (d['Continent'] == 'Europe')).sum())

fig = px.choropleth(d, locations='CCA3', color='Shown', hover_name='Country',
                    hover_data={'CCA3': False, 'Shown': False, 'Change': ':.1f'},
                    color_continuous_scale=[[0, '#B35806'], [0.5, '#F7F7F7'], [1, '#0072B2']],
                    range_color=(-80, 80), template='plotly_white')
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth',
                showcountries=True, countrycolor='white')
fig.update_layout(
    title=f"{n_shrink} countries and territories shrank between 2000 and 2022 – {n_eu} of them in Europe – "
          f"while Africa and the Middle East grew fastest<br>"
          f"<sup>Population change 2000–2022 (%), orange = shrank, blue = grew, colours capped at ±80 %</sup>",
    coloraxis_colorbar=dict(title='% change', thickness=12, len=0.6,
                            tickvals=[-80, -40, 0, 40, 80], ticktext=['−80', '−40', '0', '+40', '≥ +80']),
    margin=dict(l=0, r=0, b=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** **Treemap (World → Continent → Country), sized by 2022 population.**

- The question is about *part-to-whole* at **two hierarchical levels** in a single figure → treemap. Area encodes population, nesting encodes continent → country, so both levels are visible at once, which a bar chart or pie chart cannot do.
- Colour = continent (the same colour-blind-safe palette as in the other charts), so the top level can be read even for countries too small to be labelled. Tiles show the country name and its share of the world; tiny countries stay unlabelled but are available on hover.
- No legend or axes are needed, and white tile borders keep the figure clean.

In [11]:
total = df['2022 Population'].sum()
asia_share = df.loc[df['Continent'] == 'Asia', '2022 Population'].sum() / total
ci_share = df.loc[df['Country'].isin(['China', 'India']), '2022 Population'].sum() / total

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map={**CONT_COLORS, '(?)': '#E5E5E5'},
                 template='plotly_white')
fig.update_traces(textinfo='label+percent root', marker=dict(line=dict(width=1, color='white')),
                  root_color='#E5E5E5', hovertemplate='%{label}<br>%{value:,.0f} people<br>%{percentRoot:.1%} of world<extra></extra>')
fig.update_layout(
    title=f"Asia is home to {asia_share:.0%} of the world's {total/1e9:.1f} billion people – "
          f"China and India alone make up {ci_share:.0%}<br>"
          f"<sup>2022 population: tile area = population, colour = continent</sup>",
    margin=dict(l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** **100 % stacked bar chart, one bar per year.**

- The question is about *shares of a whole that always sums to 100 %*, changing over time → 100 % stacked bars. Normalising removes the effect of total world growth (3.7 → 8.0 billion), so only the *share* changes remain visible.
- Years are treated as categories (equal-width bars) because only 8 snapshot years exist; the percentages are written inside the bars so the reader does not need to estimate from an axis.
- Africa (orange accent) sits at the bottom so that its growth is easy to follow from a common baseline; Asia, Europe and the grouped "Rest of the world" use the same colours as in the other charts. Segments are labelled directly at the right edge instead of using a legend.

In [12]:
g = df_long.copy()
g['Group'] = g['Continent'].where(g['Continent'].isin(['Africa', 'Asia', 'Europe']), 'Rest of the world')
s = g.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
s['Share'] = s['Population'] / s.groupby('Year')['Population'].transform('sum') * 100
s['Year'] = s['Year'].astype(str)

stack = ['Africa', 'Europe', 'Rest of the world', 'Asia']          # bottom → top
colors = {'Africa': AFRICA, 'Asia': CONT_COLORS['Asia'], 'Europe': CONT_COLORS['Europe'],
          'Rest of the world': '#B8B8B8'}
years = sorted(s['Year'].unique())

fig = px.bar(s, x='Year', y='Share', color='Group', color_discrete_map=colors,
             category_orders={'Group': stack, 'Year': years},
             text=s['Share'].map(lambda v: f'{v:.0f}%'), template='plotly_white')
fig.update_traces(textposition='inside', insidetextanchor='middle', textfont=dict(color='#222', size=12))
fig.update_traces(textfont_color='white', selector=dict(name='Asia'))

# direct labels right of the last bar
last = s[s['Year'] == years[-1]].set_index('Group')['Share']
cum = 0
for grp in stack:
    fig.add_annotation(x=years[-1], y=cum + last[grp] / 2, xshift=45, xanchor='left', showarrow=False,
                       text=grp, font=dict(size=12, color='#B07800' if grp == 'Africa' else '#444'))
    cum += last[grp]

a70 = s[(s['Year'] == '1970') & (s['Group'] == 'Africa')]['Share'].iat[0]
a22 = last['Africa']
e70 = s[(s['Year'] == '1970') & (s['Group'] == 'Europe')]['Share'].iat[0]
e22 = last['Europe']
fig.update_layout(
    title=f"Africa's share of world population has risen from {a70:.0f}% to {a22:.0f}%, "
          f"Europe's has fallen from {e70:.0f}% to {e22:.0f}%, Asia's barely moved<br>"
          f"<sup>Share of world population by region, 1970–2022</sup>",
    barmode='stack', showlegend=False, margin=dict(r=130), bargap=0.25)
fig.update_xaxes(title=None, showgrid=False)
fig.update_yaxes(visible=False)
fig.show()